# EJERCICIO SQL: CERVEZAS

---

## 1. Tablas de la Base de Datos

### CERVEZAS
| CodC | Envase | Capacidad | Stock |
| :--- | :--- | :--- | :--- |
| 01 | Botella | 0.2 | 3600 |
| 02 | Botella | 0.33 | 1200 |
| 03 | Lata | 0.33 | 2400 |
| 04 | Botella | 1 | 288 |
| 05 | Barril | 60 | 30 |

### BARES
| CodB | Cif | Nombre | Localidad |
| :--- | :--- | :--- | :--- |
| 001 | 11111111X | Stop | Villa Botijo |
| 002 | 22222222Y | Las Vegas | Villa Botijo |
| 003 | - | Club Social | Las Ranas |
| 004 | 33333333Z | Otra Ronda | La Esponja |

### EMPLEADOS
| CodE | Nombre | Sueldo |
| :--- | :--- | :--- |
| 1 | Prudencio Caminero | 120000 |
| 2 | Vicente Merario | 110000 |
| 3 | Valentin Siempre | 100000 |

### REPARTO
| CodE | CodB | CodC | Fecha | Cantidad |
| :--- | :--- | :--- | :--- | :--- |
| 1 | 001 | 01 | 10/21/05 | 240 |
| 1 | 001 | 02 | 10/21/05 | 48 |
| 1 | 002 | 03 | 10/22/05 | 60 |
| 1 | 004 | 05 | 10/22/05 | 4 |
| 2 | 002 | 03 | 10/22/05 | 48 |
| 2 | 002 | 05 | 10/23/05 | 2 |
| 2 | 004 | 01 | 10/23/05 | 480 |
| 2 | 004 | 02 | 10/24/05 | 72 |
| 3 | 003 | 03 | 10/24/05 | 48 |
| 3 | 003 | 04 | 10/25/05 | 20 |

---

## 2. Enunciados de los Ejercicios

1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.
2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.
3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.
4. Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.
5. Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella. 
6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.
7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.
8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.
9. Subir un 5% el sueldo del empleado que más días haya trabajado.
10. Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.

In [1]:
import sqlite3
import pandas as pd

# Primero creamos la base de datos. Con sqlite3 podemos crear una base de datos temporal en la RAM del ordenador.
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [2]:
# creamos las tablas
cursor.executescript("""

CREATE TABLE CERVEZAS (
CodC VARCHAR(5) PRIMARY KEY, 
Envase VARCHAR(10), 
Capacidad FLOAT, 
Stock INT
);

CREATE TABLE BARES (
CodB VARCHAR(5) PRIMARY KEY, 
Cif VARCHAR(10), 
Nombre VARCHAR(25), 
Localidad VARCHAR(25)
);

CREATE TABLE EMPLEADOS (
CodE INT PRIMERY KEY, 
Nombre VARCHAR(25), 
Sueldo INT
);

CREATE TABLE REPARTO (
CodE INT,
CodB VARCHAR(5), 
CodC VARCHAR(5), 
Fecha DATE, 
Cantidad INT
);

""")

# Metemos los datos en las tablas

cursor.executescript("""
 
INSERT INTO CERVEZAS VALUES ('01','Botella',0.2,3600), ('02','Botella',0.33,1200), 
                            ('03','Lata',0.33,2400), ('04','Botella',1,288), ('05','Barril',60,30);
                            
INSERT INTO BARES VALUES ('001','11111111X','Stop','Villa Botijo'), ('002','22222222Y','Las Vegas','Villa Botijo'),
                         ('003','-','Club Social','Las Ranas'), ('004','33333333Z','Otra Ronda','La Esponja');
                         
INSERT INTO EMPLEADOS VALUES (1,'Prudencio Caminero',120000), (2,'Vicente Merario',110000), (3,'Valentin Siempre',100000);

INSERT INTO REPARTO VALUES (1,'001','01','2005-10-21',240), (1,'001','02','2005-10-21',48),
                           (1,'002','03','2005-10-22',60), (1,'004','05','2005-10-22',4),
                           (2,'002','03','2005-10-22',48), (2,'002','05','2005-10-23',2),
                           (2,'004','01','2005-10-23',480), (2,'004','02','2005-10-24',72),
                           (3,'003','03','2005-10-24',48), (3,'003','04','2005-10-25',20);
""")

In [5]:
# 1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005

# Así se harían las queries una por una, pero tambien podemos definir una función para ser mas eficientes

query1 = """
SELECT DISTINCT E.Nombre 
FROM EMPLEADOS E 
JOIN REPARTO R ON E.CodE = R.CodE 
JOIN BARES B ON R.CodB = B.CodB
WHERE B.Nombre = 'Stop' AND R.Fecha BETWEEN '2005-10-17' AND '2005-10-23'
"""

df1 = pd.read_sql_query(query1, conn)

print(df1)

               Nombre
0  Prudencio Caminero


In [6]:
# Definimos una funcion que nos haga todas las queries y les asigne un titulo

def run_query(title, sql):
    print(f"\n--- {title} ---")
    print(pd.read_sql_query(sql, conn))
    

In [7]:
run_query("2. Bares con Botella < 1L", """
SELECT DISTINCT B.Cif, B.Nombre 
FROM BARES B 
JOIN REPARTO R ON B.CodB = R.CodB 
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' AND C.Capacidad < 1
ORDER BY B.Localidad
""")

run_query("3. Repartos de Prudencio", """
SELECT B.Nombre AS Bar, C.Envase, C.Capacidad, R.Fecha, R.Cantidad
FROM REPARTO R
JOIN EMPLEADOS E ON R.CodE = E.CodE
JOIN BARES B ON R.CodB = B.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE E.Nombre = 'Prudencio Caminero'
""")

run_query("4. Bares Botella 0.2 o 0.33", """
SELECT DISTINCT B.Nombre 
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' AND C.Capacidad IN (0.2, 0.33)
""")

run_query("5. Empleados en Stop y Las Vegas (Botella)", """
SELECT E.Nombre FROM EMPLEADOS E
JOIN REPARTO R ON E.CodE = R.CodE JOIN BARES B ON R.CodB = B.CodB JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE B.Nombre = 'Stop' AND C.Envase = 'Botella'
INTERSECT
SELECT E.Nombre FROM EMPLEADOS E
JOIN REPARTO R ON E.CodE = R.CodE JOIN BARES B ON R.CodB = B.CodB JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE B.Nombre = 'Las Vegas' AND C.Envase = 'Botella'
""")

run_query("6. Viajes fuera de Villa Botijo", """
SELECT E.Nombre, COUNT(*) AS NumViajes
FROM EMPLEADOS E
JOIN REPARTO R ON E.CodE = R.CodE
JOIN BARES B ON R.CodB = B.CodB
WHERE B.Localidad != 'Villa Botijo'
GROUP BY E.Nombre
""")

run_query("7. Bar con más litros totales", """
SELECT B.Nombre, B.Localidad, SUM(R.Cantidad * C.Capacidad) AS TotalLitros
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
GROUP BY B.CodB
ORDER BY TotalLitros DESC LIMIT 1
""")

run_query("8. Bares que compraron todos los tipos de Botella < 1L", """
SELECT B.Nombre
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' AND C.Capacidad < 1
GROUP BY B.Nombre
HAVING COUNT(DISTINCT R.CodC) = (SELECT COUNT(*) FROM CERVEZAS WHERE Envase = 'Botella' AND Capacidad < 1)
""")

cursor.execute("""
UPDATE EMPLEADOS 
SET Sueldo = Sueldo * 1.05 
WHERE CodE = (
    SELECT CodE FROM REPARTO 
    GROUP BY CodE ORDER BY COUNT(DISTINCT Fecha) DESC LIMIT 1
)
""")
run_query("9. Sueldo actualizado", "SELECT * FROM EMPLEADOS")

cursor.execute("""
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
VALUES (
    (SELECT CodE FROM EMPLEADOS WHERE Nombre = 'Vicente Merario'),
    (SELECT CodB FROM BARES WHERE Nombre = 'Stop'),
    (SELECT CodC FROM CERVEZAS WHERE Envase = 'Lata' LIMIT 1),
    '2005-10-26', 
    48
)
""")
run_query("10. Reparto insertado (Vicente en Stop)", "SELECT * FROM REPARTO WHERE Fecha = '2005-10-26'")


--- 2. Bares con Botella < 1L ---
         Cif      Nombre
0  33333333Z  Otra Ronda
1  11111111X        Stop

--- 3. Repartos de Prudencio ---
          Bar   Envase  Capacidad       Fecha  Cantidad
0        Stop  Botella       0.20  2005-10-21       240
1        Stop  Botella       0.33  2005-10-21        48
2   Las Vegas     Lata       0.33  2005-10-22        60
3  Otra Ronda   Barril      60.00  2005-10-22         4

--- 4. Bares Botella 0.2 o 0.33 ---
       Nombre
0        Stop
1  Otra Ronda

--- 5. Empleados en Stop y Las Vegas (Botella) ---
Empty DataFrame
Columns: [Nombre]
Index: []

--- 6. Viajes fuera de Villa Botijo ---
               Nombre  NumViajes
0  Prudencio Caminero          1
1    Valentin Siempre          2
2     Vicente Merario          2

--- 7. Bar con más litros totales ---
       Nombre   Localidad  TotalLitros
0  Otra Ronda  La Esponja       359.76

--- 8. Bares que compraron todos los tipos de Botella < 1L ---
       Nombre
0  Otra Ronda
1        Stop

--- 

In [8]:
# Cerramos la conexión a la base de datos

conn.close()